In [ ]:
import cv2
import numpy as np
import onnxruntime as ort
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg
import ipywidgets
from IPython.display import display

# ===== KONFIG =====
MODEL_PATH = "yolov4_1_3_224_224_static.onnx"
NAMES_PATH = "coco.names"
INPUT_SIZE = 224
CONF_THRESHOLD = 0.3  # Podnieś, jeśli masz za dużo "fałszywych" ramek

# ===== KLASY =====
with open(NAMES_PATH, "r") as f:
    CLASSES = [line.strip() for line in f.readlines()]
    #CLASSES = ["Twoja_Klasa"] # Możesz też wpisać na sztywno, skoro jest jedna

# ===== MODEL (Włączamy TensorRT dla prędkości) =====
providers = [
    ('TensorrtExecutionProvider', {
        'device_id': 0,
        'trt_fp16_enable': True,
        'trt_engine_cache_enable': True,
        'trt_engine_cache_path': './'
    }),
    'CUDAExecutionProvider'
]

session = ort.InferenceSession(MODEL_PATH, providers=providers)
input_name = session.get_inputs()[0].name

print(f"Model załadowany. Klasa: {CLASSES[0]}")

# ===== KAMERA (Dodano capture_fps) =====
camera = CSICamera(width=224, height=224, capture_device=0, capture_fps=30)
camera.running = True

image_widget = ipywidgets.Image(format='jpeg', width=224, height=224)
display(image_widget)

try:
    while True:
        frame = camera.value
        if frame is None:
            continue

        # ===== PREPROCESS =====
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = img.transpose((2, 0, 1)).astype(np.float32) / 255.0
        blob = np.expand_dims(img, axis=0)

        # ===== INFERENCJA =====
        # ===== INFERENCJA =====
        # ===== INFERENCJA =====
        outputs = session.run(None, {input_name: blob})
        
        # Wyciągamy dane (zakładając standardowy format YOLOv4 ONNX)
        boxes = np.squeeze(outputs[0])  # (N, 4) -> x, y, w, h lub x1, y1, x2, y2
        scores = np.squeeze(outputs[1]) # (N, 80) -> wyniki dla każdej klasy

        # --- FILTR NA PERSON (Index 0 w COCO) ---
        target_class_id = 0 # W COCO 0 to zazwyczaj 'person'
        
        # Pobieramy wyniki tylko dla ludzi
        # Jeśli scores ma wymiar (N, 80), to scores[:, 0] to kolumna 'person'
        if len(scores.shape) > 1:
            person_scores = scores[:, target_class_id]
        else:
            # Jeśli model ma tylko jedną klasę (Twoje własne trenowanie), używamy całości
            person_scores = scores

        # Szukamy najlepszej detekcji człowieka
        best_idx = np.argmax(person_scores)
        conf = float(person_scores[best_idx])

        # ===== RYSUJ TYLKO JEŚLI TO CZŁOWIEK I PEWNOŚĆ JEST OK =====
        if conf > CONF_THRESHOLD:
            x1, y1, x2, y2 = boxes[best_idx]

            # Skalowanie do wymiaru 224x224
            ix1 = int(max(0, x1 * 224))
            iy1 = int(max(0, y1 * 224))
            ix2 = int(min(224, x2 * 224))
            iy2 = int(min(224, y2 * 224))

            # Rysowanie (tylko dla person)
            label_text = f"PERSON {conf:.2f}"
            cv2.rectangle(frame, (ix1, iy1), (ix2, iy2), (0, 255, 0), 2) # Niebieski dla ludzi
            cv2.putText(frame, label_text, (ix1, iy1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

        # ===== WYŚWIETLANIE =====
        image_widget.value = bgr8_to_jpeg(frame)

except KeyboardInterrupt:
    camera.running = False
    print("Zatrzymano.")